# Konu 11 — Müşterinin söylediklerini unutmayan bir hafıza

Yarın Sayfa Kahve'ye yeni bir afiş gidiyor. Son üç haftada müşteri sana WhatsApp'tan, e-postayla ve toplantıda
bir sürü şey söyledi; jüride hocalar da aynı eleştiriyi iki kez yaptı. Hepsi bir yerde yazılı ama dağınık.
Afişi yaparken hangisini hatırlayacaksın?

Bugün bir **müşteri hafızası** kuruyoruz: notlarını anlamına göre arayan ve cevabını yalnız o notlara dayandıran
bir yardımcı. Not ekledikçe hafıza büyür, cevap da değişir.

Bunun adı **RAG**. Açık kitap sınavı gibi düşün: model ezberinden cevap vermiyor, önüne konan sayfalara bakıyor.
Doğru sayfaları bulup önüne koymak bizim işimiz. (İngilizce *retrieval-augmented generation*: "bulup getirerek üretmek".)

Hücreleri yukarıdan aşağı, sırayla çalıştır: hücreye tıkla, **Shift + Enter**. Isınma anahtarsız; Adım 1'den
sonrası `anahtar.txt` ister. Defterin tamamı 12 istek gönderir (10 vektör, 2 metin), günlük ücretsiz kotanın küçük bir parçası.

## Isınma — Notlarını oku

`veri/notlar.txt` senin not defterin: her satır bir not, başında kimin, nereden, ne zaman söylediği yazıyor.
Müşterinin mesajları, jüri notları, kendi notların ve işle ilgisi olmayanlar (fatura, matbaa) karışık duruyor.

In [ ]:
notlar = []
with open("veri/notlar.txt", encoding="utf-8") as dosya:
    for satir in dosya:
        notlar.append(satir.strip())

print(len(notlar))
print(notlar[0])

16 not (Adım 4'ü daha önce çalıştırdıysan fazlası). Yarın afiş için fotoğraf çekeceksin ve aklına bir soru geldi:
**"Masada oturan bir öğrencinin portresini çekeyim mi?"**

**Tahmin et:** notlarda bu soruya cevap veren bir not var mı? Hangisi?

Bir dosyada bir şey ararken çoğumuz kelimeyle ararız (Ctrl + F). Deneyelim:

In [ ]:
aranan = "portre"
for not_metni in notlar:
    if aranan in not_metni:
        print(not_metni)

Hiçbir şey çıkmadı. `aranan`'ı `"öğrenci"` yap ve hücreyi yeniden çalıştır. Gelen not portreyle ilgili mi?

Doğru not şu: "Fotoğraflarda insan yüzü olmasın; masalar, kitaplar ve eller olabilir."
Anlamı sorunla aynı ama ortak kelimesi yok: sen "portre" dedin, not "yüz" diyor. Kelime araması iki türlü yanılır:
ortak kelime yoksa doğru notu bulamaz, ortak kelime varsa konusu başka bir notu getirir ("öğrenci").

Notları **anlamına göre** aramamız gerekiyor. Konu 4'ten biliyorsun: anlamı yakın iki metnin vektörleri aynı yöne bakar.

## Adım 1 — Hafızayı vektöre çevir

Hafızaya iki şey giriyor: müşterinin yazılı marka kılavuzu (`veri/kilavuz.txt`, her satır bir kural) ve senin
topladığın notlar. Yardımcı dosya `rag_araclari.py`'de hazır fonksiyonlar var: `vektor_al` ve `benzerlik` Konu 4'te
yazdıkların, `modele_sor` Konu 5'tekinin benzeri. `notlari_oku` ısınmadaki dört satırın işini yapar.

In [ ]:
from rag_araclari import notlari_oku, hafizayi_kur, hafizada_ara
from rag_araclari import vektor_al, benzerlik, modele_sor

kilavuz = notlari_oku("veri/kilavuz.txt")
print(len(kilavuz))

`kilavuz + notlar` iki listeyi uç uca ekler: 14 kural ve 16 not, 30 metin. `hafizayi_kur` hepsini **tek istekte**
vektör modeline gönderir ve Konu 4'teki `vektorler_sozlugu` gibi bir sözlük kurar: anahtar metin, değer onun vektörü.

In [ ]:
hafiza = hafizayi_kur(kilavuz + notlar)
print(len(hafiza))

## Adım 2 — Soruya en yakın 3 not

Isınmadaki soruyu bu kez anlamıyla arıyoruz: sorunun vektörünü alıyor, hafızadaki her metinle benzerliğini
ölçüyoruz (Konu 4'teki gibi). **Tahmin et:** en yüksek puanı hangi metin alacak?

In [ ]:
soru = "Masada oturan bir öğrencinin portresini çekeyim mi?"
soru_vektoru = vektor_al(soru)

for metin in hafiza:
    vektor = hafiza[metin]
    puan = benzerlik(soru_vektoru, vektor)
    kisa = round(puan, 2)
    print(kisa, metin)

30 satır, her metnin yanında puanı. En büyük üç puanı gözünle bul: üçü de fotoğrafla ilgili mi?
Fatura ve matbaa notları kaç puan aldı?

Bunu her soruda gözle yapmak yorucu. Yardımcıdaki `hafizada_ara` yukarıdaki hücrenin aynısını yapar, sonra
puanları büyükten küçüğe dizip ilk `kac` metni verir (Konu 4'teki `most_common` ile).

In [ ]:
bulunanlar = hafizada_ara(soru, hafiza, 3)
for metin in bulunanlar:
    print(metin)

Başka bir soru. **Tahmin et:** hangi not gelecek? Sonra `soru`'yu değiştirip yeniden çalıştır (her deneme 1 istek):
"Başlıkla paragraf neredeyse aynı büyüklükte, sorun olur mu?", "Afişi matbaaya ne zaman göndermeliyim?" (matbaa notu
bu kez gelir mi?) ve kendi sorun.

In [ ]:
soru = "Hikâyede yazıyı en alta koyabilir miyim?"
bulunanlar = hafizada_ara(soru, hafiza, 3)
for metin in bulunanlar:
    print(metin)

Tasarım sorularında fatura notu gelmez: anlamı başka yöne bakıyor, puanı düşük kalıyor. Matbaa sorusunda ise
matbaa notu gelir. Hafızada "ilgisiz" not yok; yalnız o soruyla ilgisi olmayan not var.

## Adım 3 — Afiş müşterinin kurallarına uyuyor mu?

Konu 10'daki hat afişteki yazıları okumuştu. Afişi yeniden okutup kota harcamayalım, herkeste aynı metin olsun diye
afişin yazılarını `veri/afis_metni.txt`'ye koyduk.

In [ ]:
with open("veri/afis_metni.txt", encoding="utf-8") as dosya:
    afis_metni = dosya.read()

print(afis_metni)

Açık kitap sınavının iki yarısı var. Önce sayfaları bul: hafızada afiş metniyle ilgili notları arıyoruz.

In [ ]:
arama = "Etkinlik afişinin metninde hangi bilgiler olmalı, hangi sözcükler olmamalı?"
bulunanlar = hafizada_ara(arama, hafiza, 5)
for metin in bulunanlar:
    print(metin)

Gelen beş metni oku. Hepsi bu afişle mi ilgili? Arama her zaman istediğin sayıda metin getirir: soruya en yakın
beşini, ilgisi zayıf olsa bile. Afişte "kafemizde" yazıyor; müşterinin "kafe" notu bu beşin içinde mi?

Sonra sayfaları modelin önüne koy. `yonerge_kur` soruyu ve bulunan notları tek bir metinde birleştirir; bu Python'un
işi, modele istek gitmez. İlk satır kuralımız: **yalnız bu notlara dayan; notlarda yoksa "notlarda yok" de.**
Bu kural olmazsa model boşlukları kendi tahminiyle doldurur (Konu 5'teki halüsinasyon, yani uydurma). Dayandığı notu
yazmasını da istiyoruz: cevabı kontrol edebilmek için.

In [ ]:
def yonerge_kur(soru, bulunanlar):
    yonerge = "Yalnız aşağıdaki notlara dayan. Notlarda yoksa 'notlarda yok' de.\n"
    yonerge = yonerge + "Kısa maddelerle cevap ver; her maddede dayandığın notu yaz.\n\nNOTLAR:\n"
    for metin in bulunanlar:
        yonerge = yonerge + "- " + metin + "\n"
    yonerge = yonerge + "\nSORU: " + soru
    return yonerge

In [ ]:
soru = "Afişin metni:\n" + afis_metni + "\n\n"
soru = soru + "Notları tek tek bu afişle karşılaştır. Her not için bir madde yaz: afiş bu nota uyuyor mu, uymuyor mu? "
soru = soru + "Uymuyorsa afişteki hangi söz ya da hangi eksik bilgi? Afişle ilgisi olmayan bir notsa 'ilgisiz' de."
yonerge = yonerge_kur(soru, bulunanlar)
print(yonerge)

Modelin önüne konan sayfalar bunlar. **Tahmin et:** model hangi sorunları bulacak? Afişte saat var mı?
"Kafemizde" için bir şey diyecek mi?

In [ ]:
cevap = modele_sor(yonerge, 0.2)
print(cevap)

Cevabını oku; yanındakinin cevabı biraz farklı olabilir.

- Her maddenin yanında bir not var mı? O not gerçekten yukarıdaki listede mi?
- Notlarda olmayan bir şey söyledi mi? Söylediyse kuralın dışına çıkmış demektir.

**Arama kaçırırsa model de kaçırır.** "Kafe" notu listede yoksa model "kafemizde"nin sorun olduğunu bilemez: önüne
konmayan notu göremez. **Dene:** aşağıdaki hücre aynı hafızada başka bir cümleyle arıyor. Çalıştır; sonra cümleyi
kendin değiştir ya da `5`'i `8` yap. Senin çıktında ne değişti? "Kafe" notu geldi mi? İlk aramada gelip burada
gelmeyen bir not var mı?

In [ ]:
deneme = "Afiş metninde hangi sözcükler kullanılmamalı?"
bulunanlar = hafizada_ara(deneme, hafiza, 5)
for metin in bulunanlar:
    print(metin)

Hangi notların geleceğini arama cümlesi belirler; tek bir cümle afişin bütün sorularını kapsamayabilir. Gerçek işte
iki ayrı arama yapıp bulunanları `+` ile birleştirebilirsin (Adım 1'deki `kilavuz + notlar` gibi). Adım 4 yine ilk
aramayı (`arama`) kullanıyor.

## Adım 4 — Hafıza büyür

30 Ekim, müşteriden yeni bir WhatsApp mesajı: *"Etkinlik afişlerinde 'ücretsiz' sözcüğü de olmasın, kampanya gibi
duruyor; 'çay ikramı' demek yeter."* Bir yere yazmazsan üç hafta sonra unutursun. Hafızaya ekliyoruz: önce dosyaya
(yarın da orada olsun), sonra vektörüyle sözlüğe (şimdi aranabilsin).

`"a"` dosyayı açar ve sonuna ekler (*append*). `"w"` olsaydı dosyayı silip baştan yazardı: bütün notların giderdi.
`"\n"` satır sonu: her not kendi satırında dursun.

In [ ]:
yeni_not = "Sayfa Kahve, WhatsApp, 30 Ekim: Etkinlik afişlerinde 'ücretsiz' sözcüğü de olmasın, kampanya gibi duruyor; 'çay ikramı' demek yeter."

with open("veri/notlar.txt", "a", encoding="utf-8") as dosya:
    dosya.write(yeni_not + "\n")

hafiza[yeni_not] = vektor_al(yeni_not)
print(len(hafiza))

**Tahmin et:** Adım 3'teki aramanın aynısı. Yeni not ilk 5'e girer mi?

In [ ]:
bulunanlar = hafizada_ara(arama, hafiza, 5)
for metin in bulunanlar:
    print(metin)

Aynı soru, aynı kural. **Tahmin et:** cevap değişecek mi?

In [ ]:
yonerge = yonerge_kur(soru, bulunanlar)
cevap = modele_sor(yonerge, 0.2)
print(cevap)

Adım 3'teki cevapla karşılaştır: "Ücretsiz çay ikramı" satırı şimdi sorun mu? Model bunu hangi nota dayandırdı?
Hiçbir kodu değiştirmedik; hafızaya yalnız bir satır ekledik. Konu 12'de bu işi asistanın "hatırla" aracı yapacak.

**Kendi elinle:** bir jüri notu ekle. Aşağıdaki hücrede `benim_notum`'u kendi aldığın bir eleştiriyle değiştir
(aynı biçimde: `Jüri, tarih: not`). Altına, notu dosyaya ve hafızaya ekleyen satırları **kendin** yaz: yukarıdaki
ekleme hücresine bakabilirsin, yalnız `yeni_not` yerine `benim_notum` yaz.

In [ ]:
benim_notum = "Jüri, 30 Ekim: Görselle başlık birbirine yapışık; aralarında nefes bırak."

Şimdi notunu bulacak bir soru sor; `soru`'yu kendi notuna göre değiştir. Notu eklediysen ilk üçte görmelisin.

In [ ]:
soru = "Görselle başlık birbirine çok yakın, sorun olur mu?"
bulunanlar = hafizada_ara(soru, hafiza, 3)
for metin in bulunanlar:
    print(metin)

**Dürüst not:** 30 metin için hepsini yönergeye koymak da olurdu; "kafe" notu da kaçmazdı. Arama, notlar dönem
boyunca ve birkaç müşteriyle yüzlere çıkınca gerekir. Ama hangi notun cevabı belirlediğini görmek her zaman işe yarar.

## Bonus — Kendi müşterinin hafızası

Konu 8'de seçtiğin müşteri için beş not yaz: müşterinin istekleri ve aldığın eleştiriler. Örneği sil, kendininkini
koy. (3 istek: notlar, soru, cevap.)

In [ ]:
musterim = [
    "Defne Çiçekçilik, WhatsApp, 20 Ekim: Gönderilerde pembe kullanmayalım; yeşil ve krem kalsın.",
    "Defne Çiçekçilik, toplantı notu, 22 Ekim: Fiyat yazmayalım, 'mağazaya sorun' diyelim.",
    "Defne Çiçekçilik, e-posta, 24 Ekim: Logomuz her görselde sağ alt köşede dursun.",
    "Jüri, 23 Ekim: Fotoğraflar çok kalabalık; tek bir çiçeğe odaklan.",
    "Kendi notum, 26 Ekim: Hikâyelerde yazıyı en fazla iki satırda tut.",
]
musteri_hafizasi = hafizayi_kur(musterim)
print(len(musteri_hafizasi))

In [ ]:
soru = "Anneler Günü gönderisinde buketin fiyatını yazayım mı?"
bulunanlar = hafizada_ara(soru, musteri_hafizasi, 2)
yonerge = yonerge_kur(soru, bulunanlar)
cevap = modele_sor(yonerge, 0.2)
print(cevap)